# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, math, time, subprocess
import numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F

MSSV = "2A202602451"
REPO_URL = "https://github.com/SxAinsworth/K4-Track4-Day1-Neuralnetwork.git"   # fork của mình (có code đã push)
IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")

if IN_COLAB:
    # Kernel Colab (kể cả khi mở từ VS Code) chạy trên máy chủ: lấy code + dữ liệu bằng git,
    # rồi chuyển cwd vào submission_<MSSV>/code để mọi đường dẫn tương đối giống hệt khi chạy local.
    COLAB_REPO = "/content/K4-Track4-Day1-Neuralnetwork"
    if not os.path.isdir(COLAB_REPO):
        subprocess.run(["git", "clone", REPO_URL, COLAB_REPO], check=True)
    else:
        subprocess.run(["git", "-C", COLAB_REPO, "pull", "--ff-only"], check=True)
    os.chdir(f"{COLAB_REPO}/submission_{MSSV}/code")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REPO_ROOT = "../.."     # gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
OUT_DIR   = ".."        # submission_<MSSV>/: nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "results"), exist_ok=True)

# Kiểm tra trước khi đọc dữ liệu: repo phải chứa data/ và kết quả phải nằm trong thư mục nộp.
repo_root_abs = os.path.abspath(REPO_ROOT)
out_dir_abs = os.path.abspath(OUT_DIR)
print(f"REPO_ROOT -> {repo_root_abs}")
print(f"OUT_DIR   -> {out_dir_abs}")
assert os.path.isdir(os.path.join(repo_root_abs, "data")), "REPO_ROOT không trỏ tới gốc repo có data/."
assert os.path.basename(out_dir_abs) == f"submission_{MSSV}", "OUT_DIR không trỏ tới thư mục nộp."


def push_outputs(msg="Update results from Colab"):
    """Đẩy figures/, results/, xlsx, csv, json từ runtime Colab về GitHub (máy local chỉ cần git pull).
    Cần GitHub token (fine-grained, quyền Contents: read/write trên repo fork); token không được lưu vào notebook."""
    if not IN_COLAB:
        print("Đang chạy local: kết quả đã nằm sẵn trên máy, không cần push.")
        return
    from getpass import getpass
    token = os.environ.get("GH_TOKEN") or getpass("GitHub token: ")
    os.environ["GH_TOKEN"] = token                      # chỉ giữ trong phiên runtime hiện tại
    git = ["git", "-C", COLAB_REPO, "-c", "user.name=SxAinsworth",
           "-c", "user.email=SxAinsworth@users.noreply.github.com"]
    subprocess.run(git + ["add", f"submission_{MSSV}"], check=True)
    if subprocess.run(git + ["diff", "--cached", "--quiet"]).returncode == 0:
        print("Không có gì mới để push.")
        return
    subprocess.run(git + ["commit", "-q", "-m", msg], check=True)
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    subprocess.run(git + ["pull", "--rebase", "-q", url, "main"], check=True)
    subprocess.run(git + ["push", "-q", url, "HEAD:main"], check=True)
    print("Đã push. Trên máy local chạy: git pull")


from data import prepare_data, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# Chia train/eval theo metadata (chỉ chạy script nếu chưa có file .npz)
if not os.path.exists(f"{REPO_ROOT}/data/processed/eval.npz"):
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)

set_seed(42)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
assert len(data["X_tr"]) == 371_847 and len(data["X_val"]) == 92_962 and len(data["X_eval"]) == 116_203

# Tỉ lệ lớp ở 3 tập phải gần như bằng nhau (phân tầng)
print("\nlớp   train(%)   val(%)   eval(%)")
props = [torch.bincount(data[k], minlength=7).float().cpu() / len(data[k]) * 100 for k in ("y_tr", "y_val", "y_eval")]
for c in range(7):
    print(f"{c:>3d}   {props[0][c]:7.3f}  {props[1][c]:7.3f}  {props[2][c]:7.3f}")

# 10 cột số trên X_tr sau chuẩn hoá: mean ≈ 0, std ≈ 1; 44 cột nhị phân giữ nguyên 0/1
num = data["X_tr"][:, :10]
print("\nmean 10 cột số (X_tr):", np.round(num.mean(0).cpu().numpy(), 4))
print("std  10 cột số (X_tr):", np.round(num.std(0).cpu().numpy(), 4))
print("cột nhị phân chỉ gồm 0/1:", bool(torch.isin(data["X_tr"][:, 10:], torch.tensor([0.0, 1.0], device=device)).all()))

In [ ]:
# Kiểu dữ liệu và khoảng nhãn của các tensor dùng chung cho mọi thí nghiệm
for k in ("X_tr", "y_tr", "X_val", "y_val", "X_eval", "y_eval"):
    t = data[k]
    extra_info = f"nhãn {t.min().item()}..{t.max().item()}" if k.startswith("y") else ""
    print(f"{k:<7s} {str(tuple(t.shape)):<12s} {str(t.dtype):<14s} {t.device}  {extra_info}")
assert all(data[k].dtype == torch.float32 and data[k].shape[1] == 54 for k in ("X_tr", "X_val", "X_eval"))
assert all(data[k].dtype == torch.int64 for k in ("y_tr", "y_val", "y_eval"))

# eval_row_id phải gắn đúng thứ tự mẫu eval: đủ 116 203 id, không trùng, và đối chiếu ngẫu nhiên vài dòng với CSV gốc
import pandas as pd
rid = data["eval_row_id"]
assert len(rid) == len(data["X_eval"]) == 116_203 and len(np.unique(rid)) == len(rid)
raw = pd.read_csv(f"{REPO_ROOT}/data/covtype.csv.gz")
check = np.random.default_rng(0).choice(len(rid), size=200, replace=False)
csv_rows = raw.iloc[rid[check]]
X_eval_raw = csv_rows.drop(columns="Cover_Type").to_numpy(np.float32)
X_eval_back = data["X_eval"][check].cpu().numpy().copy()
X_eval_back[:, :10] = X_eval_back[:, :10] * np.where(data["std"] > 0, data["std"], 1) + data["mean"]   # bỏ chuẩn hoá
assert np.allclose(X_eval_back, X_eval_raw, atol=1e-2), "X_eval lệch với CSV theo row_id"
assert (csv_rows["Cover_Type"].to_numpy() - 1 == data["y_eval"][check].cpu().numpy()).all(), "nhãn eval lệch với CSV"
print("eval_row_id: 200 dòng ngẫu nhiên khớp CSV gốc (đặc trưng và nhãn) ✓")
del raw

# iterate_batches: mỗi epoch đi qua MỌI mẫu train đúng một lần; lô cuối nhỏ hơn được giữ lại
sizes, seen = [], []
g = torch.Generator().manual_seed(0)
for xb, yb in iterate_batches(data["X_tr"], data["y_tr"], 512, generator=g):
    sizes.append(len(xb))
print(f"batch 512: {len(sizes)} lô/epoch, lô cuối = {sizes[-1]} mẫu, tổng = {sum(sizes)}")
assert sum(sizes) == len(data["X_tr"]) and sizes[-1] == len(data["X_tr"]) % 512

**Nhận xét Part 0:**
- Kích thước khớp GUIDE: train còn 371 847, val 92 962, eval 116 203. Tỉ lệ lớp ở ba tập gần như trùng nhau nhờ phân tầng. Phép tách val dùng **seed 42 cố định** cho mọi thí nghiệm; khi đổi seed huấn luyện để đo nhiễu, seed tách val vẫn giữ nguyên.
- Mốc "luôn đoán lớp đa số" (lớp 1) trên val = 0,4876: mọi mô hình phải vượt mức này.
- **Vì sao chỉ dùng X_tr để tính mean/std?** Mean/std là tham số của pipeline, cũng "học" từ dữ liệu như trọng số. Nếu tính trên val hoặc eval, thông tin của tập dùng để đánh giá rò rỉ vào bước tiền xử lý, nên điểm val/eval không còn là ước lượng trung thực cho dữ liệu chưa thấy. Áp dụng *cùng* mean/std của train lên val/eval thì được phép (giống khi triển khai: dữ liệu mới chỉ được biến đổi bằng thống kê đã có).
- 44 cột nhị phân (Wilderness_Area, Soil_Type) giữ nguyên 0/1; nếu cột số nào có std = 0 thì chỉ trừ mean (tránh chia cho 0).
- **Lô cuối:** 371 847 = 726 × 512 + 135, nên lô cuối có 135 mẫu. Mình giữ lô này (không `drop_last`) để mỗi epoch dùng hết dữ liệu; nó chỉ chiếm 1/727 số bước nên ảnh hưởng không đáng kể.

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
# 1a–1b. Dựng M-base, kiểm tra số tham số và shape qua từng lớp
set_seed(42)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model)
assert n_params == EXPECTED_PARAMS[(256, 128)] == 47_879, n_params
print(model)
print("số tham số:", n_params)

h = torch.randn(8, 54, device=device)
print(f"\n{'đầu vào':<54s} {tuple(h.shape)}")
for layer in model.net:
    h = layer(h)
    print(f"{str(layer):<54s} {tuple(h.shape)}")
assert h.shape == (8, 7)

In [ ]:
# 1b (tiếp). Kiểm tra cấu trúc, không chỉ shape
linears = [m for m in model.modules() if isinstance(m, nn.Linear)]
assert [(m.in_features, m.out_features) for m in linears] == [(54, 256), (256, 128), (128, 7)]
assert all(m.bias is not None for m in linears), "mọi Linear phải có bias"
assert isinstance(model.net[-1], nn.Linear), "lớp cuối phải là Linear (logit thô, không softmax)"
assert not any(isinstance(m, (nn.Softmax, nn.LogSoftmax, nn.BatchNorm1d, nn.LayerNorm)) for m in model.modules())
print("Linear:", [(m.in_features, m.out_features) for m in linears], "| bias: có ở mọi lớp | không softmax/BatchNorm/LayerNorm")

# Hai kiến trúc tuỳ chọn cũng phải khớp số tham số quy định
for hid in [(512, 256), (256, 128, 64)]:
    n = count_params(MLP(hidden=hid))
    assert n == EXPECTED_PARAMS[hid], (hid, n)
    print(f"MLP{hid}: {n:,} tham số ✓")

# Dropout (khi q > 0) chỉ nằm ngay sau ReLU của lớp ẩn, không ở đầu vào hay trên logit
m_drop = MLP(hidden=(256, 128), dropout=0.3)
kinds = [type(l).__name__ for l in m_drop.net]
print("q = 0.3:", " → ".join(kinds))
assert kinds == ["Linear", "ReLU", "Dropout", "Linear", "ReLU", "Dropout", "Linear"]
assert all(l.p == 0.3 for l in m_drop.net if isinstance(l, nn.Dropout))

# He thực sự được áp dụng: std(W) ≈ sqrt(2/n_in), bias = 0; mặc định của nn.Linear chỉ cho std ≈ 1/sqrt(3·n_in)
for i, m in enumerate(linears, 1):
    print(f"W{i}: std = {m.weight.std().item():.4f} | He kỳ vọng sqrt(2/{m.in_features}) = {math.sqrt(2 / m.in_features):.4f}"
          f" | mặc định nn.Linear ≈ {1 / math.sqrt(3 * m.in_features):.4f} | bias toàn 0: {bool((m.bias == 0).all())}")
    assert abs(m.weight.std().item() / math.sqrt(2 / m.in_features) - 1) < 0.1 and (m.bias == 0).all()

In [ ]:
# 1c-1. Loss bước 0 trên toàn bộ val (eval mode, chưa cập nhật bước nào); mốc chẩn đoán ln 7
model.eval()
with torch.no_grad():
    logits0 = model(data["X_val"])
    loss0 = F.cross_entropy(logits0, data["y_val"]).item()
print(f"loss bước 0 trên val = {loss0:.4f}   |   ln 7 = {math.log(7):.4f}   |   chênh = {loss0 - math.log(7):+.4f}")
print("std kích hoạt bước 0 (sau ReLU1, ReLU2, logit):", [round(v, 3) for v in activation_stats(model, data["X_val"][:4096])])

# Chẩn đoán độ lệch: logit có lớn không, và có lệch hệ thống về một lớp không?
print("trung bình logit theo lớp trên val:", np.round(logits0.mean(0).cpu().numpy(), 3))
print("tỉ lệ dự đoán theo lớp ở bước 0:  ", np.round(torch.bincount(logits0.argmax(1), minlength=7).cpu().numpy() / len(logits0), 3))
# Phép thử đối chứng (không đổi baseline): thu nhỏ W3 100 lần thì logit ≈ 0, softmax ≈ đều, loss phải ≈ ln 7
with torch.no_grad():
    z = logits0 - model.net[-1].bias      # bias = 0 nên đây chính là W3·h
    loss_small = F.cross_entropy(z * 0.01, data["y_val"]).item()
print(f"đối chứng: logit × 0.01 -> loss = {loss_small:.4f} (≈ ln 7)")

In [ ]:
# 1c-2. Quá khớp 20 mẫu: tắt dropout, Adam lr=1e-2, 500 bước full-batch -> loss phải về gần 0
set_seed(42)
tiny = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
idx = torch.randperm(len(data["X_tr"]), generator=torch.Generator().manual_seed(0))[:20].to(device)
xb, yb = data["X_tr"][idx], data["y_tr"][idx]
opt = torch.optim.Adam(tiny.parameters(), lr=1e-2)
tiny_losses, tiny_accs = [], []
for step in range(500):
    tiny.train()
    opt.zero_grad()
    logits = tiny(xb)
    loss = F.cross_entropy(logits, yb)
    loss.backward()
    opt.step()
    tiny_losses.append(loss.item())                                   # loss trước bước cập nhật này
    tiny_accs.append((logits.argmax(1) == yb).float().mean().item())
first100 = next(k for k, a in enumerate(tiny_accs) if a == 1.0)
tiny.eval()
with torch.no_grad():
    acc20 = (tiny(xb).argmax(1) == yb).float().mean().item()
print(f"loss bước 0 = {tiny_losses[0]:.4f}, loss cuối = {tiny_losses[-1]:.2e}, accuracy trên 20 mẫu = {acc20:.2f}")
print("nhãn của 20 mẫu:", torch.bincount(yb, minlength=7).tolist(), f"| đạt 100% từ bước {first100}")
assert tiny_losses[-1] < 1e-2 and acc20 == 1.0

import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.semilogy(tiny_losses, label="CE trên 20 mẫu")
ax1.axhline(math.log(7), ls="--", c="gray", label="ln 7")
ax1.set_xlabel("bước"); ax1.set_ylabel("loss (thang log)"); ax1.set_title("Loss"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(tiny_accs, c="C2")
ax2.set_xlabel("bước"); ax2.set_ylabel("accuracy"); ax2.set_ylim(0, 1.05); ax2.set_title("Accuracy trên 20 mẫu"); ax2.grid(alpha=0.3)
fig.suptitle("Quá khớp 20 mẫu: M-base, He, dropout 0, Adam lr=1e-2, full-batch")
fig.tight_layout()
plt.show()

In [ ]:
# 1d. Gradient có chảy tới mọi tham số? (một lần backward trên một lô val, model mới khởi tạo)
model.train()
model.zero_grad()
F.cross_entropy(model(data["X_val"][:512]), data["y_val"][:512]).backward()
names = ["W1", "b1", "W2", "b2", "W3", "b3"]
for name, (pname, p) in zip(names, model.named_parameters()):
    assert p.grad is not None and p.grad.norm().item() > 0, f"{name} không có gradient"
    print(f"{name} ({pname:<12s}) shape {str(tuple(p.shape)):<11s} ‖grad‖ = {p.grad.norm().item():.4e}")
model.zero_grad()

**Nhận xét Part 1:**
- **Cấu trúc:** M-base có đúng 47 879 tham số; ba Linear (54→256, 256→128, 128→7) đều có bias, lớp cuối ra logit thô, không có softmax, BatchNorm hay LayerNorm. Một lô (8, 54) cho logits (8, 7). M-wide (161 287) và M-deep (55 687) cũng khớp bảng. Khi q > 0, Dropout chỉ nằm ngay sau ReLU của hai lớp ẩn.
- **Khởi tạo He đã được áp dụng thật:** std(W) đo được 0,194 / 0,088 / 0,130, khớp sqrt(2/n_in) = 0,193 / 0,088 / 0,125, và khác xa mức mặc định của `nn.Linear` (≈ 0,079 / 0,036 / 0,051). Bias đều bằng 0.
- **Loss bước 0 trên val = 2,378, cao hơn ln 7 = 1,946 khoảng 0,43.** ln 7 chỉ đúng khi mọi logit gần bằng nhau (softmax đều). Với He, lớp cuối cũng có Var(W3) = 2/128, nên logit có std ≈ 0,65 chứ không ≈ 0. Hơn nữa, vì các kích hoạt sau ReLU luôn ≥ 0 và có trung bình dương, W3·h tạo ra một độ lệch **chung cho mọi mẫu** (trung bình logit theo lớp từ −0,42 đến +0,94). Kết quả là ở bước 0 model đoán lớp 4 (Aspen, chỉ 1,6% dữ liệu) cho 68% số mẫu, nên CE cao hơn mức "đoán đều". Phép đối chứng xác nhận nguyên nhân: thu nhỏ logit 100 lần cho loss 1,948 ≈ ln 7. Đây là hệ quả của cách khởi tạo, không phải lỗi dữ liệu hay nhãn (chuẩn hoá đã kiểm tra ở Part 0). Baseline vẫn giữ He theo quy định; chủ đề *khởi tạo* ở Part 3 sẽ so sánh trực tiếp (ví dụ `normal` std 0,01 cho logit gần 0, nên loss bước 0 sẽ ≈ ln 7).
- **Quá khớp 20 mẫu:** với Adam lr = 1e-2, full-batch, dropout 0, accuracy trên 20 mẫu đạt 100% từ bước 5, loss giảm từ 2,37 xuống 1,3e-6 sau 500 bước. Như vậy nhãn, logit → CE, `zero_grad` và danh sách tham số truyền cho optimizer đều đúng. Phép thử này chỉ cho thấy model *đủ năng lực và vòng cập nhật hoạt động*, chưa nói gì về khả năng tổng quát hoá.
- **Gradient:** sau một lần `backward()`, cả 6 tham số W1, b1, W2, b2, W3, b3 đều có gradient khác None và khác 0 (‖grad‖ từ 0,38 đến 2,34), nên không có lớp nào bị "đứt" gradient.

*(Các số trên đo ở seed 42. Nếu chạy trên GPU mà số lẻ cuối lệch chút ít thì cập nhật theo output của ô bên trên.)*

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# TODO: chọn lr cho baseline bằng VAL (chạy vài giá trị, ví dụ quanh 0.01 -> 0.1); ghi bảng lr -> val_macro_f1
# TODO: cfg = {**DEFAULT_CFG, "lr": <lr đã chọn>}
# TODO: chạy baseline với >= 2 seed (khuyến khích 3): result = run_experiment({**cfg, "seed": s, "exp_id": f"base-s{s}"}, data)
# TODO: với mỗi result: save_result(result, f"{OUT_DIR}/results"); plot_run(result, f"{OUT_DIR}/figures/{exp_id}.png")
# TODO: tính trung bình ± độ lệch chuẩn (val_macro_f1, val_acc) qua các seed -> ngưỡng nhiễu 2σ

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Mẫu một thí nghiệm. Lặp lại / bọc trong vòng for qua danh sách cfg bạn tự thiết kế.
# TODO: exp_cfg = {**cfg, "exp_id": "...", "group": "...", "description": "...", <chỉ đổi MỘT yếu tố>}
# TODO: result = run_experiment(exp_cfg, data)
# TODO: save_result(...); plot_run(...)   # mỗi thí nghiệm một ảnh figures/<exp_id>.png
# TODO: in summary (best_val_loss, val_acc, val_macro_f1, ...) và so với baseline + ngưỡng nhiễu 2σ

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [ ]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [ ]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary